In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df=pd.read_csv('Spotify_Music.csv')
df.head()

,slno,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
df.drop(columns=['track_id', 'album_name', 'track_name'],inplace=True)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   slno              114000 non-null  int64  
 1   artists           113999 non-null  object 
 2   popularity        114000 non-null  int64  
 3   duration_ms       114000 non-null  int64  
 4   explicit          114000 non-null  bool   
 5   danceability      114000 non-null  float64
 6   energy            114000 non-null  float64
 7   key               114000 non-null  int64  
 8   loudness          114000 non-null  float64
 9   mode              114000 non-null  int64  
 10  speechiness       114000 non-null  float64
 11  acousticness      114000 non-null  float64
 12  instrumentalness  114000 non-null  float64
 13  liveness          114000 non-null  float64
 14  valence           114000 non-null  float64
 15  tempo             114000 non-null  float64
 16  time_signature    11

In [5]:
df.isnull().any()

slno                False
artists              True
popularity          False
duration_ms         False
explicit            False
danceability        False
energy              False
key                 False
loudness            False
mode                False
speechiness         False
acousticness        False
instrumentalness    False
liveness            False
valence             False
tempo               False
time_signature      False
track_genre         False
dtype: bool

In [6]:
df[df.isnull().any(axis=1)]

,slno,artists,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,65900,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [7]:
df.dropna(inplace=True)

In [8]:
df.drop(columns='slno',inplace=True)

In [9]:
x=df.drop(columns='track_genre')
y=df.track_genre

In [10]:
obj_cols=x.select_dtypes(include='object').columns
obj_cols

Index(['artists'], dtype='object')

In [11]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [12]:
x[obj_cols].nunique()

artists    31437
dtype: int64

In [13]:
preprocessing=ColumnTransformer(
    transformers=[
        ('encoder', TargetEncoder(), obj_cols)
    ],
    remainder='passthrough'
)

In [14]:
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',RandomForestClassifier(n_estimators=150, max_depth=25, min_samples_leaf=5, random_state=42))
    ]
)

In [15]:
main_pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
ypred_train=main_pipeline.predict(xtrain)

In [17]:
print(classification_report(ytrain, ypred_train))

                   precision    recall  f1-score   support

         acoustic       0.77      0.89      0.83       787
         afrobeat       0.89      0.95      0.92       797
         alt-rock       0.49      0.48      0.49       785
      alternative       0.48      0.53      0.50       816
          ambient       0.76      0.83      0.79       803
            anime       0.74      0.85      0.79       807
      black-metal       0.88      0.93      0.90       790
        bluegrass       0.87      0.95      0.91       795
            blues       0.60      0.66      0.63       786
           brazil       0.66      0.60      0.63       803
        breakbeat       0.89      0.92      0.90       801
          british       0.78      0.62      0.69       786
         cantopop       0.84      0.94      0.89       807
    chicago-house       0.95      0.94      0.95       794
         children       0.93      0.96      0.94       786
            chill       0.76      0.79      0.77       

In [18]:
ypred_test=main_pipeline.predict(xtest)

In [19]:
print(classification_report(ytest,ypred_test))

                   precision    recall  f1-score   support

         acoustic       0.39      0.36      0.38       213
         afrobeat       0.63      0.60      0.61       203
         alt-rock       0.06      0.06      0.06       215
      alternative       0.12      0.14      0.13       184
          ambient       0.49      0.50      0.49       197
            anime       0.32      0.29      0.30       193
      black-metal       0.71      0.60      0.65       210
        bluegrass       0.61      0.72      0.66       205
            blues       0.23      0.19      0.21       214
           brazil       0.12      0.10      0.11       197
        breakbeat       0.62      0.47      0.54       199
          british       0.29      0.12      0.17       214
         cantopop       0.58      0.58      0.58       193
    chicago-house       0.67      0.46      0.55       206
         children       0.79      0.71      0.75       214
            chill       0.32      0.30      0.31       